In [1]:
%%html
<img src="img/lora.png",width=400,height=200>

# 手写 LoRA 层

In [8]:
import torch
import torch.nn as nn
import math

class LinearLoRALayer(nn.Module):
    """LoRA线性层的核心实现
    """
    def __init__(self,
            input_dim,
            output_dim,
            rank=8,
            lora_alpha=16,
            dropout=0.1,
            merge=False,
            debug=False
        ):
        super().__init__()

        assert rank > 0, "rank must be large than zero!"

        self.input_dim = input_dim
        self.output_dim = output_dim
        self.merge = merge
        self.rank = rank

        # 注意这里的self.linear.weight.data shape是[output_dim, input_dim], X*W^T
        self.linear = nn.Linear(input_dim, output_dim, bias=False)
        '''
        ΔW=B⋅A
        ΔW 是对原始权重 的更新量，因此 ΔW 必须与 W
        形状完全一致，才能进行加法合并（无论是训练时叠加还是推理时 merge）。
        '''
        # 定义LoRA可学习低秩矩阵 B, shape=[output_dim, rank]
        self.lora_b = nn.Parameter(torch.zeros(output_dim, rank))

        # 定义LoRA可学习低秩矩阵 A, shape=[rank, input_dim]
        # 记住一点，给输入 X 下投影那个矩阵要随机初始化，上投影那个矩阵要全0初始化
        self.lora_a = nn.Parameter(torch.zeros(rank, input_dim))
        nn.init.normal_(self.lora_a, mean=0, std=0.01)

        if debug:
            print("lora_a initial value: ", self.lora_a)
            print("lora_b initial value: ", self.lora_b)

        # 定义delta权重更新的系数
        self.scale = lora_alpha / rank

        # 冻结 linear weight，不可训练
        self.linear.weight.requires_grad = False

        # 定义dropout
        self.dropout = nn.Dropout(dropout)

        # 如果要做推理，而不是训练，一般都要做 merge， 加快推理速度
        if merge:
            self.merge_lora()

    def forward(self, hidden_states):
        # hidden_states shape = [batch, seq_len, input_dim]
        # lora_b shape = [output_dim, rank]
        # lora_a shape = [rank, input_dim]
        # lora_b * lora_a = [output_dim, input_dim]
        # output: hidden_states * W^T = [batch, seq_len, input_dim] * [input_dim, output_dim] = [batch, seq_len, output_dim] 
        if self.merge:
            output = self.linear(hidden_states)
        else:
            output = self.linear(hidden_states) + self.scale * (hidden_states @ (self.lora_b @ self.lora_a).T)

        return self.dropout(output)

    def merge_lora(self):
        self.linear.weight.data += self.scale * (self.lora_b @ self.lora_a)

    def unmerge_lora(self):
        self.linear.weight.data -= self.scale * (self.lora_b @ self.lora_a)

# 测试代码

In [17]:
# 超参数设置
batch_size = 4
seq_len = 64
input_dim = 128
output_dim = 256
rank = 8
lora_alpha = 16
dropout = 0.1

# 构造输入
x = torch.randn(batch_size, seq_len, input_dim)

torch.manual_seed(42)
# 不做 Merge (For 训练阶段)
lora_linear = LinearLoRALayer(
    input_dim=input_dim,
    output_dim=output_dim,
    rank=rank,
    lora_alpha=lora_alpha,
    dropout=dropout,
    merge=False,
    debug=True)

output = lora_linear(x)
print(f"Output shape:  {output.shape}")
print("output: ", output[0])

lora_a initial value:  Parameter containing:
tensor([[-0.0078, -0.0036, -0.0139,  ..., -0.0059, -0.0011, -0.0010],
        [-0.0046, -0.0128,  0.0124,  ...,  0.0096, -0.0174, -0.0088],
        [ 0.0029,  0.0119, -0.0040,  ..., -0.0095,  0.0154,  0.0010],
        ...,
        [ 0.0059, -0.0151,  0.0039,  ..., -0.0092, -0.0051,  0.0034],
        [ 0.0102, -0.0156, -0.0035,  ...,  0.0032,  0.0010,  0.0173],
        [-0.0081,  0.0018,  0.0120,  ...,  0.0019,  0.0084, -0.0211]],
       requires_grad=True)
lora_b initial value:  Parameter containing:
tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], requires_grad=True)
Output shape:  torch.Size([4, 64, 256])
output:  tensor([[ 0.3245, -0.0211, -0.0557,  ...,  0.0167,  0.4732, -0.5732],
        [-0.6968,  0.8584, -0.2818,  ...,  0.1427,  0.38

In [18]:
# 做 Merge (For 测试阶段)
torch.manual_seed(42)
lora_linear_merged = LinearLoRALayer(
    input_dim=input_dim,
    output_dim=output_dim,
    rank=rank,
    lora_alpha=lora_alpha,
    dropout=dropout,
    merge=True,
    debug=False)

output_merged = lora_linear_merged(x)
print(f"Output shape:  {output_merged.shape}")
print("output: ", output_merged[0])

Output shape:  torch.Size([4, 64, 256])
output:  tensor([[ 0.3245, -0.0211, -0.0557,  ...,  0.0167,  0.4732, -0.5732],
        [-0.6968,  0.8584, -0.2818,  ...,  0.1427,  0.3847,  0.6276],
        [ 0.2556, -0.6734, -0.6315,  ..., -1.1083, -0.8230, -0.9921],
        ...,
        [ 0.6652, -1.0924,  0.1108,  ...,  0.1752,  0.4753,  0.6252],
        [ 0.0000,  0.0227,  0.6872,  ..., -0.4397,  0.2620, -1.1251],
        [ 0.6071,  0.0000, -0.9939,  ..., -0.2122,  0.9427,  0.2344]])


In [14]:
# 测试权重 Merge/UnMerge
lora_linear.merge_lora()

#merge weights
torch.manual_seed(42)
merge_output = lora_linear(x)

# unmerge weights
lora_linear.unmerge_lora()
torch.manual_seed(42)
unmerge_output = lora_linear(x)

print("merge and unmerge error: ", torch.max(torch.abs(unmerge_output - merge_output)).item())



merge and unmerge error:  0.0
